In [1]:
# %pip install -e ./orchestrator
# ! ansible-galaxy collection install -r orchestrator/ansible/requirements.yaml

# import orchestrator
# import shutil
# assert shutil.which("ansible-playbook")
# print(orchestrator.__file__)

Obtaining file:///Users/zimabrou/Projects/5g_blueprint/orchestrator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for orchestrator (pyproject.toml) ... done
  Created wheel for orchestrator: filename=orchestrator-0.0.1-py3-none-any.whl size=2360 sha256=71a35af5b811dd8cbe8de252072570c8fac47bc4d1bcfd18dfcc7cc29e215163
  Stored in directory: /private/var/folders/4j/8j63qg_x4y93h7cg9lzd1n90k524rf/T/pip-ephem-wheel-cache-n6t8t8mw/wheels/8e/39/20/1d5153775ce91dfa36ac18b3e74b6017595bf40312f8f6f2a3
Successfully built orchestrator
  Attempting uninstall: orchestrator
    Found existing installation: orchestrator 0.0.1
    Uninstalling orchestrator-0.0.1:
      Successfully uninstalled orchestrator-0.0.1

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
N

In [1]:
from orchestrator.models import MetaModel

# Choose exactly one input model:
# RIC_METAMODEL_FILE = "metamodel_flexric.yaml"  # FlexRIC
RIC_METAMODEL_FILE = "metamodel_osc_ric.yaml"  # O-RAN SC

metamodel = MetaModel.from_yaml_file(RIC_METAMODEL_FILE)

In [2]:
from orchestrator.models import DeploymentModel

deployment_model = DeploymentModel()
deployment_model.to_yaml_file("deployment_model_1.yaml")
deployment_model.to_yaml_file("deployment_model_full_1.yaml", exclude_defaults=False)

In [3]:
from orchestrator.deployment import BaseBlueprintManagement

reconciled_metamodel, reconciled_deployment_model = BaseBlueprintManagement(metamodel=metamodel, deployment_model=deployment_model).output()

In [4]:
reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_1.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_1.yaml")

In [5]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [6]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ******************************

In [7]:
metamodel, deployment_model = prov.output()

deployment_model.to_yaml_file("deployment_model_2.yaml")
metamodel.to_yaml_file("metamodel_2.yaml")

# checkpoint 1: provisioned

In [8]:
from orchestrator.deployment import ProvisioningManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_2.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_2.yaml")

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [9]:
prov.get_merged(output="yaml", path="merged_model.yaml")

In [10]:
prov.get_ansible_inventory(path="inventory.ini")

In [11]:
from orchestrator.deployment import K8SSetupManagement

k8s = K8SSetupManagement(metamodel=metamodel, deployment_model=deployment_model)

In [12]:
await k8s.deploy()


PLAY [node1] *******************************************************************

PLAY [node3] *******************************************************************

PLAY [node4] *******************************************************************

PLAY [node2] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************
[WARNING]: Host 'node3' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node3]

TASK [k8

In [13]:
metamodel, deployment_model = k8s.output()

deployment_model.to_yaml_file("deployment_model_3.yaml")
metamodel.to_yaml_file("metamodel_3.yaml")

# checkpoint 2: k8s ready

In [14]:
from orchestrator.deployment import CoreManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_3.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_3.yaml")

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [15]:
reconciled_metamodel, reconciled_deployment_model = core.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_2.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_2.yaml")

In [16]:
core.mutable_metamodel.to_yaml_file("mutatated_metamodel.yaml", exclude_defaults=False)

In [17]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-database' chart] *****

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-udr' chart] **********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-nrf' chart] **********

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-smf' chart] **********

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-udm' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-amf' chart] **********

PLAY [node1] **

In [18]:
metamodel, deployment_model = core.output()

deployment_model.to_yaml_file("deployment_model_4.yaml")
metamodel.to_yaml_file("metamodel_4.yaml")

# checkpoint 3: core deployed

In [ ]:
from orchestrator.deployment import RICManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_4.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_4.yaml")

# Deploys the RIC declared in the active metamodel:
# implementation: oai for FlexRIC, or oran_sc for O-RAN SC.
ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await ric.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oran-sc-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oran-sc-e2-termination' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oran-sc-xapp-onboarder' chart] ***

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oran-sc-a1-mediator' chart] ***

TASK [helm/prepare : Create Helm build directory for 'oran-sc-application-manager' chart] ***

PLAY [node1] *******************************************************************

PLAY [node1] ************************************************

In [7]:
metamodel, deployment_model = ric.output()

deployment_model.to_yaml_file("deployment_model_5.yaml")
metamodel.to_yaml_file("metamodel_5.yaml")

# checkpoint 4: ric deployed

In [11]:
from orchestrator.deployment import RANManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_5.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_5.yaml")

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [12]:
reconciled_metamodel, reconciled_deployment_model = ran.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_3.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_3.yaml")

In [ ]:
# await ran.remove(force=True)

In [13]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=4    unreach

In [14]:
metamodel, deployment_model = ran.output()

deployment_model.to_yaml_file("deployment_model_6.yaml")
metamodel.to_yaml_file("metamodel_6.yaml")

# checkpoint 5: ran deployed

In [15]:
from orchestrator.deployment import UEManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_6.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_6.yaml")

ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

In [16]:
reconciled_metamodel, reconciled_deployment_model = ue.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_4.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_4.yaml")

In [17]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue' chart] *******
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-sim-ue' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepare : Render templated Helm files for 'oai-sim-ue'] *************
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    chan

In [18]:
metamodel, deployment_model = ue.output()

deployment_model.to_yaml_file("deployment_model_7.yaml")
metamodel.to_yaml_file("metamodel_7.yaml")

In [ ]:
# await ue.remove()

In [ ]:
# await ran.remove(force=True)

In [ ]:
# await core.remove()

In [ ]:
# \await prov.unprovision()